# 01 - EDA e Ingeniería de Datos (PySpark)

**Proyecto:** Analítica de Cadena de Suministro - Starbucks Costa Rica

Este notebook realiza la **ingeniería de datos** del proyecto usando Apache Spark:

1. Inicializa una `SparkSession` en modo local.
2. Carga los CSV limpios generados por `extract_data.py`.
3. Realiza un análisis exploratorio (esquemas, conteos, nulos).
4. Transforma el histórico de formato **ancho** (wk1..wk174, 174 semanas continuas) a formato **largo** `[SKU, semana, demanda]` con `stack()`.
5. Une dimensiones de producto (costo, vida útil) y marca SKUs de campaña.
6. Persiste los resultados en **Parquet** para reutilizarlos en los modelos.

> Los CSV de entrada se producen ejecutando `python extract_data.py` desde la raíz del proyecto.


In [1]:
import os
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

if not os.environ.get("JAVA_HOME"):
    candidatos = (
        glob.glob(r"C:\Program Files\Microsoft\jdk-17*")
        + glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*")
        + glob.glob(os.path.expanduser(r"~\AppData\Local\Microsoft\jdk-17*"))
    )
    if candidatos:
        os.environ["JAVA_HOME"] = candidatos[0]
        os.environ["PATH"] = os.path.join(candidatos[0], "bin") + os.pathsep + os.environ.get("PATH", "")

_root = "." if os.path.isdir("hadoop") else ".."
_hadoop_home = os.path.abspath(os.path.join(_root, "hadoop"))
if os.path.isdir(os.path.join(_hadoop_home, "bin")):
    os.environ["HADOOP_HOME"] = _hadoop_home
    os.environ["PATH"] = os.path.join(_hadoop_home, "bin") + os.pathsep + os.environ.get("PATH", "")

print("JAVA_HOME  :", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = (
    SparkSession.builder
    .appName("Starbucks-EDA-DataEngineering")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


JAVA_HOME  : C:\Program Files\Microsoft\jdk-17.0.19.10-hotspot\
HADOOP_HOME: D:\Documents\bigdataproject\hadoop


Spark version: 4.1.2


In [2]:
ROOT = "." if os.path.isdir("data") else ".."
DATA_DIR = os.path.join(ROOT, "data")
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
os.makedirs(PARQUET_DIR, exist_ok=True)

df_hist = spark.read.csv(os.path.join(DATA_DIR, "historico_semanal.csv"), header=True, inferSchema=True)
df_inf = spark.read.csv(os.path.join(DATA_DIR, "inf.csv"), header=True, inferSchema=True)
df_camp = spark.read.csv(os.path.join(DATA_DIR, "refuerzo_campanas.csv"), header=True, inferSchema=True)
df_bio = spark.read.csv(os.path.join(DATA_DIR, "bio.csv"), header=True, inferSchema=True)

print("historico_semanal:", df_hist.count(), "filas")
print("inf            :", df_inf.count(), "filas")
print("refuerzo_campanas:", df_camp.count(), "filas")
print("bio            :", df_bio.count(), "filas")

historico_semanal: 436 filas


inf            : 471 filas


refuerzo_campanas: 51 filas


bio            : 33 filas


## Exploración de esquemas y muestra

In [3]:
df_hist.select("SKU", "DESCRIPCION", "PROVEEDOR", "PROM_AJUST", "EXIST_W11", "TRANSITO", "wk1", "wk2", "wk52").show(5, truncate=40)

+-------+----------------------------------------+------------------------------+------------------+---------+--------+---+---+----+
|    SKU|                             DESCRIPCION|                     PROVEEDOR|        PROM_AJUST|EXIST_W11|TRANSITO|wk1|wk2|wk52|
+-------+----------------------------------------+------------------------------+------------------+---------+--------+---+---+----+
| 100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|               0.0|      5.0|     0.0|0.0|0.0| 0.0|
|1007648|                   GAVETAS PARA MASTRENA|STARBUCKS CORPORATION Utileria|               0.0|      2.0|     0.0|0.0|0.0| 0.0|
| 101091|           TINTA PARA PISTOLA DE PRECIOS|STARBUCKS CORPORATION Utileria|               0.0|      0.0|     0.0|0.0|0.0| 0.0|
| 103242|                               CUBO PLEX|STARBUCKS CORPORATION Utileria|1.6500000000000001|     25.0|     9.0|2.0|0.0| 0.0|
|  10508|                      PISTOLA DE PRECIOS|STARBUCKS CORPORATI

In [4]:
key_cols = ["SKU", "PROVEEDOR", "PROM_AJUST", "EXIST_W11", "TRANSITO"]
df_hist.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in key_cols
]).show()

+---+---------+----------+---------+--------+
|SKU|PROVEEDOR|PROM_AJUST|EXIST_W11|TRANSITO|
+---+---------+----------+---------+--------+
|  0|       16|         0|        0|       0|
+---+---------+----------+---------+--------+



## Paso 1 - Transformación de formato ancho a largo (data melting)

El histórico tiene **174 semanas continuas** como columnas (`wk1..wk174`). Para alimentar modelos de ML necesitamos un formato largo `[SKU, semana, demanda]`. Lo logramos con la función `stack()` de Spark SQL.


In [5]:
import re as _re
semanas_cols = sorted(
    [c for c in df_hist.columns if _re.fullmatch(r"wk\d+", c)],
    key=lambda c: int(c[2:]),
)
N_SEMANAS = len(semanas_cols)
print(f"Semanas historicas detectadas: {N_SEMANAS}")

stack_parts = ", ".join([f"{i}, cast({col} as double)" for i, col in enumerate(semanas_cols, start=1)])
stack_expr = f"stack({N_SEMANAS}, {stack_parts}) as (semana, demanda)"

df_largo = (
    df_hist
    .select("SKU", "DESCRIPCION", "PROVEEDOR", F.expr(stack_expr))
    .withColumn("semana", F.col("semana").cast(IntegerType()))
    .withColumn("demanda", F.col("demanda").cast("double"))
)

print("Filas en formato largo:", df_largo.count())
df_largo.show(8)


Semanas historicas detectadas: 174


Filas en formato largo: 75864


+------+--------------------+--------------------+------+-------+
|   SKU|         DESCRIPCION|           PROVEEDOR|semana|demanda|
+------+--------------------+--------------------+------+-------+
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     1|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     2|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     3|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     4|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     5|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     6|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     7|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     8|    0.0|
+------+--------------------+--------------------+------+-------+
only showing top 8 rows


In [6]:
df_largo = df_largo.fillna({"demanda": 0.0})
df_largo = df_largo.filter(F.col("SKU").isNotNull())
print("Nulos restantes en demanda:", df_largo.filter(F.col("demanda").isNull()).count())

Nulos restantes en demanda: 0


## Paso 3 - Enriquecimiento: costo del producto y bandera de campaña

In [7]:
df_largo = df_largo.withColumn("SKU", F.col("SKU").cast("string"))
df_inf = df_inf.withColumn("SKU", F.col("SKU").cast("string"))
df_camp = df_camp.withColumn("SKU", F.col("SKU").cast("string"))

inf_costo = df_inf.select("SKU", "COSTO_UNITARIO", "VIDA_UTIL", "CS_POR_PALLET")

# La serie comienza el lunes de la semana ISO 1 de 2023. Se marca cada SKU
# solo durante las semanas que se traslapan con las fechas de vigencia.
semana_fechada = (
    df_largo.select("SKU", "semana").distinct()
    .withColumn(
        "fecha_inicio",
        F.date_add(F.to_date(F.lit("2023-01-02")), ((F.col("semana") - 1) * 7).cast("int")),
    )
    .withColumn("fecha_fin", F.date_add(F.col("fecha_inicio"), 6))
)
campanas_fechadas = (
    df_camp.select("SKU", "LAUNCH_DATE", "END_DATE")
    .withColumn("fecha_lanzamiento", F.to_date("LAUNCH_DATE"))
    .withColumn("fecha_fin_campana", F.coalesce(F.to_date("END_DATE"), F.to_date("LAUNCH_DATE")))
    .filter(F.col("fecha_lanzamiento").isNotNull())
)
semana_campana = (
    semana_fechada.alias("w")
    .join(
        campanas_fechadas.alias("c"),
        (F.col("w.SKU") == F.col("c.SKU"))
        & (F.col("c.fecha_lanzamiento") <= F.col("w.fecha_fin"))
        & (F.col("c.fecha_fin_campana") >= F.col("w.fecha_inicio")),
        "left",
    )
    .select(
        F.col("w.SKU").alias("SKU"),
        F.col("w.semana").alias("semana"),
        F.when(F.col("c.fecha_lanzamiento").isNotNull(), 1).otherwise(0).alias("es_campana"),
    )
    .groupBy("SKU", "semana")
    .agg(F.max("es_campana").alias("es_campana"))
)

df_modelo = (
    df_largo
    .join(inf_costo, on="SKU", how="left")
    .join(semana_campana, on=["SKU", "semana"], how="left")
    .fillna({"es_campana": 0, "VIDA_UTIL": 0.0, "CS_POR_PALLET": 0.0})
)

df_modelo.show(8, truncate=40)
print("Semanas SKU-campaña marcadas:", df_modelo.filter(F.col("es_campana") == 1).count())

+------+------+----------------------------------------+------------------------------+-------+--------------+---------+-------------+----------+
|   SKU|semana|                             DESCRIPCION|                     PROVEEDOR|demanda|COSTO_UNITARIO|VIDA_UTIL|CS_POR_PALLET|es_campana|
+------+------+----------------------------------------+------------------------------+-------+--------------+---------+-------------+----------+
|100396|     1|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|    0.0|          2.74|      0.0|         15.0|         0|
|100396|     2|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|    0.0|          2.74|      0.0|         15.0|         0|
|100396|     3|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|    0.0|          2.74|      0.0|         15.0|         0|
|100396|     4|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|    0.0|          2.74|      0.0|    

Semanas SKU-campaña marcadas: 1


## Paso 4 — Valor físico, tránsito y calidad del catálogo

La valorización separa existencias físicas de mercancía en tránsito. Los SKU deben ser únicos antes de combinar inventario y costos; si hay claves duplicadas se detiene el cálculo para no inflar cantidades o dinero. Un costo desconocido permanece sin valorizar, nunca se sustituye por cero. No se suman unidades ni demanda de SKU heterogéneos ni se presenta una cobertura agregada del portafolio; la cobertura se interpreta por SKU y solo cuando su demanda es positiva.

In [ ]:
def assert_unique_sku(frame, label):
    duplicates = frame.groupBy("SKU").count().filter(F.col("count") > 1).limit(1).count()
    if duplicates:
        raise ValueError(f"{label} contiene SKU duplicados; se cancela el join para evitar inflar el valor.")


assert_unique_sku(df_hist, "historico_semanal")
assert_unique_sku(df_inf, "inf")
bad_inventory = df_hist.filter(
    F.col("EXIST_W11").isNull() | F.col("TRANSITO").isNull()
    | (F.col("EXIST_W11") < 0) | (F.col("TRANSITO") < 0)
    | F.isnan("EXIST_W11") | F.isnan("TRANSITO")
    | (F.abs("EXIST_W11") == float("inf"))
    | (F.abs("TRANSITO") == float("inf"))
)
if bad_inventory.limit(1).count():
    raise ValueError("El snapshot contiene existencias o tránsito nulos, negativos o no finitos.")
bad_cost = df_inf.filter(
    F.col("COSTO_UNITARIO").isNotNull()
    & (F.isnan("COSTO_UNITARIO") | (F.abs("COSTO_UNITARIO") == float("inf")) | (F.col("COSTO_UNITARIO") < 0))
)
if bad_cost.limit(1).count():
    raise ValueError("El costo debe ser finito y no negativo; corrija la fuente antes de valorizar.")

inv_df = (
    df_hist.select("SKU", "DESCRIPCION", "PROVEEDOR", "EXIST_W11", "TRANSITO")
    .withColumn("SKU", F.col("SKU").cast("string"))
    .join(df_inf.select("SKU", "COSTO_UNITARIO").withColumn("SKU", F.col("SKU").cast("string")), on="SKU", how="left")
    .withColumn("IP", F.col("EXIST_W11") + F.col("TRANSITO"))
    .withColumn("costo_conocido", F.col("COSTO_UNITARIO").isNotNull() & ~F.isnan("COSTO_UNITARIO"))
    .withColumn("valor_bodega_usd", F.when(F.col("costo_conocido"), F.col("EXIST_W11") * F.col("COSTO_UNITARIO")))
    .withColumn("valor_transito_usd", F.when(F.col("costo_conocido"), F.col("TRANSITO") * F.col("COSTO_UNITARIO")))
)

valor_bodega_conocido = inv_df.agg(F.sum("valor_bodega_usd").alias("v")).first()["v"]
valor_transito_conocido = inv_df.agg(F.sum("valor_transito_usd").alias("v")).first()["v"]
valor_bodega_conocido = valor_bodega_conocido or 0.0
valor_transito_conocido = valor_transito_conocido or 0.0
skus_con_existencia = inv_df.filter(F.col("EXIST_W11") > 0).count()
skus_costo_desconocido = inv_df.filter(~F.col("costo_conocido")).count()

print(f"Valor conocido de existencias físicas (USD): {valor_bodega_conocido:,.2f}")
print(f"Valor conocido de mercancía en tránsito (USD; separado): {valor_transito_conocido:,.2f}")
print(f"SKU con existencias físicas: {skus_con_existencia}; SKU sin costo conocido: {skus_costo_desconocido}")
print("La posición IP se calcula por SKU; no se agregan unidades ni demanda heterogéneas del catálogo.")

print("\nTop 10 SKU por valor de existencias físicas conocidas:")
inv_df.filter(F.col("costo_conocido")) \
    .select("SKU", "DESCRIPCION", "PROVEEDOR", "EXIST_W11", "TRANSITO", "IP", "COSTO_UNITARIO", "valor_bodega_usd", "valor_transito_usd") \
    .orderBy(F.col("valor_bodega_usd").desc()) \
    .show(10, truncate=55)

print("\nSKU con costo desconocido: no se reportan como valor cero.")
inv_df.filter(~F.col("costo_conocido")) \
    .select("SKU", "DESCRIPCION", "EXIST_W11", "TRANSITO") \
    .show(10, truncate=55)

print("\nConcentración monetaria conocida por proveedor (USD; sin sumar unidades):")
inv_df.filter(F.col("costo_conocido")).groupBy("PROVEEDOR") \
    .agg(F.sum("valor_bodega_usd").alias("valor_bodega_usd"), F.countDistinct("SKU").alias("skus_valorizados")) \
    .orderBy(F.col("valor_bodega_usd").desc()) \
    .show(5, truncate=40)


## Paso 5 — Cobertura histórica por SKU (no es pronóstico)

Esta lectura usa la demanda media histórica por SKU y la posición de inventario disponible en el archivo. No reemplaza el pronóstico ni valida que inventario y demanda tengan la misma fecha de corte. La cobertura de demanda cero queda indefinida; no se asigna un denominador artificial. La interpretación es individual por SKU, no un total del portafolio.

In [ ]:
demanda_por_sku = (
    df_largo.groupBy("SKU")
    .agg(F.avg("demanda").alias("media_historica_semanal"))
)
cobertura_por_sku = (
    inv_df.select("SKU", "DESCRIPCION", "EXIST_W11", "TRANSITO", "IP")
    .join(demanda_por_sku, on="SKU", how="left")
    .withColumn(
        "cobertura_historica_semanas",
        F.when(F.col("media_historica_semanal") > 0, F.col("IP") / F.col("media_historica_semanal")),
    )
)
print("SKU con demanda histórica cero: cobertura indefinida; validar catálogo y ventana antes de actuar.")
cobertura_por_sku.filter(F.col("media_historica_semanal") <= 0).select(
    "SKU", "DESCRIPCION", "EXIST_W11", "TRANSITO", "media_historica_semanal", "cobertura_historica_semanas"
).show(10, truncate=55)
print("SKU con menor cobertura histórica positiva (lectura exploratoria; el snapshot no tiene fecha certificada):")
cobertura_por_sku.filter(F.col("cobertura_historica_semanas").isNotNull()).orderBy(
    F.col("cobertura_historica_semanas").asc()
).select(
    "SKU", "DESCRIPCION", "EXIST_W11", "TRANSITO", "media_historica_semanal", "cobertura_historica_semanas"
).show(10, truncate=55)


## Paso 6 — Figuras descriptivas por SKU

Las series se muestran en paneles separados, con escalas propias, para no sumar ni comparar unidades de productos heterogéneos como si fueran equivalentes. El ranking solo selecciona SKU para inspección; no representa valor empresarial ni riesgo causal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CAPTURAS_DIR = os.path.join(ROOT, "informe", "img", "eda")
os.makedirs(CAPTURAS_DIR, exist_ok=True)

CAPTURAS_DIR = os.path.join(ROOT, "informe", "img", "eda")
os.makedirs(CAPTURAS_DIR, exist_ok=True)
inv_top_pd = (
    inv_df.filter(F.col("costo_conocido"))
    .select("SKU", "valor_bodega_usd")
    .orderBy(F.col("valor_bodega_usd").desc())
    .limit(12)
    .toPandas()
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(inv_top_pd["SKU"].astype(str), inv_top_pd["valor_bodega_usd"], color="#00704A")
ax.invert_yaxis()
ax.set_title("Existencias físicas: 12 SKU con mayor valor conocido")
ax.set_xlabel("USD a costo unitario conocido; tránsito excluido")
fig.tight_layout()
fig.savefig(os.path.join(CAPTURAS_DIR, "01_valor_fisico_top_skus.png"), dpi=130)
plt.close(fig)

top_skus = (
    df_largo.groupBy("SKU").agg(F.sum("demanda").alias("demanda_historica_sku"))
    .orderBy(F.desc("demanda_historica_sku"))
    .limit(8)
    .select("SKU")
)
demand_pd = (
    df_largo.join(top_skus, on="SKU", how="inner")
    .select("SKU", "semana", "demanda")
    .orderBy("SKU", "semana")
    .toPandas()
)
skus = demand_pd["SKU"].drop_duplicates().tolist()
fig, axes = plt.subplots(len(skus), 1, figsize=(11, max(8, len(skus) * 1.8)), sharex=True, sharey=False)
axes = [axes] if len(skus) == 1 else list(axes)
for ax, sku in zip(axes, skus):
    series = demand_pd[demand_pd["SKU"] == sku]
    ax.plot(series["semana"], series["demanda"], linewidth=1.1, color="#00704A")
    ax.set_title(f"SKU {sku} — escala individual", loc="left", fontsize=9)
    ax.set_ylabel("Unidades")
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Índice semanal del histórico; no fecha de corte certificada")
fig.suptitle("Demanda histórica por SKU — paneles no agregables", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(CAPTURAS_DIR, "02_demanda_por_sku.png"), dpi=130, bbox_inches="tight")
plt.close(fig)
print("Figuras descriptivas por SKU generadas en:", CAPTURAS_DIR)


## Paso 7 - Persistencia en Parquet

Guardamos las tablas procesadas en formato columnar Parquet (optimizado para Spark) para que los notebooks de modelado las reutilicen sin repetir el preprocesamiento.

In [8]:
(df_modelo.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "demanda_larga")))

(df_hist.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "historico")))
(df_inf.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "inf")))
(df_bio.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "bio")))

print("Parquet escritos en:", PARQUET_DIR)
print(os.listdir(PARQUET_DIR))

Parquet escritos en: ..\data\parquet
['bio', 'demanda_larga', 'forecast_sku', 'historico', 'inf', 'plan_reabastecimiento', 'segmentos_sku']


In [9]:
spark.stop()
print("SparkSession finalizada.")

SparkSession finalizada.
